In [37]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os

In [79]:
raw_ihs_poverty_2010 = pd.read_csv(r'd:\GG\source\householdpoverty_10.CSV')
raw_ihs_weight_2010 = pd.read_csv(r'd:\GG\source\householdweight_10.CSV')
raw_ihs_weight_2015 = pd.read_csv(r'd:\GG\source\householdweight_15.CSV')
raw_ihs_poverty_2015 = pd.read_csv(r'd:\GG\source\householdpoverty_15.CSV')
raw_dhs_2020 = pd.read_stata(r'd:\GG\source\household_19_20.DTA')
raw_findex_2021 = pd.read_csv(r'd:\GG\source\connectivity_21.csv')
raw_findex_2024 = pd.read_csv(r'd:\GG\source\connectivity_24.csv')

In [87]:
raw_dhs_2020

,hhid,hv000,hv001,hv002,hv003,hv004,hv005,hv006,hv007,hv008,...,sb113l_81,sb113l_82,sb113l_83,sb113l_84,sb113l_85,sb113l_86,sb113l_87,sb113l_88,sb113l_89,sb113l_90
0,1 1,GM7,1,1,1,1,172113,11,2019,1439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 6,GM7,1,6,2,1,172113,11,2019,1439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 10,GM7,1,10,1,1,172113,11,2019,1439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 14,GM7,1,14,1,1,172113,11,2019,1439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1 18,GM7,1,18,1,1,172113,11,2019,1439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6544,281 36,GM7,281,36,2,281,590524,2,2020,1442,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6545,281 38,GM7,281,38,1,281,590524,2,2020,1442,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6546,281 39,GM7,281,39,6,281,590524,2,2020,1442,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6547,281 41,GM7,281,41,2,281,590524,2,2020,1442,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [89]:
# aggregate economic rank
# 2010
df_2010 = pd.DataFrame()
df_2010['hid'] = raw_ihs_poverty_2010['hid']
df_2010['survey_year'] = 2010
df_2010['data_source'] = 'IHS'
df_2010['lga'] = raw_ihs_poverty_2010['lga']

weight_lookup = raw_ihs_weight_2010[['hid', 'weightslga']].drop_duplicates(subset=['hid'])
df_2010 = df_2010.merge(weight_lookup, on='hid', how='left')
df_2010['weightslga'] = df_2010['weightslga'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2010.drop_duplicates('hid').set_index('hid')['s11q2']
df_2010['hh_income'] = df_2010['hid'].astype(int).map(poverty_map)

# 2015
df_2015 = pd.DataFrame()
df_2015['hh_id'] = raw_ihs_poverty_2015['hid']
df_2015['survey_year'] = 2015
df_2015['data_source'] = 'IHS'
df_2015['eanum'] = raw_ihs_poverty_2015['eanum']

weight_lookup = raw_ihs_weight_2015[['eanum', 'hhweight']].drop_duplicates(subset=['eanum'])
df_2015 = df_2015.merge(weight_lookup, on='eanum', how='left')
df_2015['hh_weight'] = df_2015['hhweight'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2015.drop_duplicates('hid').set_index('hid')['s13q3']
df_2015['hh_income'] = df_2015['hh_id'].astype(int).map(poverty_map)

# 2020
df_2020 = pd.DataFrame()
df_2020['hh_id'] = raw_dhs_2020['hhid']
df_2020['survey_year'] = raw_dhs_2020['hv007']
df_2020['hh_income'] = raw_dhs_2020['hv270']

df_2020['data_source'] = 'DHS'
df_2020['weight'] = raw_dhs_2020['hv005']

quintile_labels = {'poorest': 1, 'poorer': 2, 'middle': 3, 'richer': 4, 'richest': 5}
df_2020['hh_income'] = df_2020['hh_income'].map(quintile_labels)

# 2021
df_2021 = pd.DataFrame()
df_2021['hh_id'] = raw_findex_2021.index.map(lambda x: f"findex_21_{x}")
df_2021['survey_year'] = 2021
df_2021['data_source'] = 'Findex'
df_2021['hh_income'] = raw_findex_2021['inc_q']
df_2021['weight'] = (raw_findex_2021['wgt']).fillna(NA)

# 2024
df_2024 = pd.DataFrame()
df_2024['hh_id'] = raw_findex_2024.index.map(lambda x: f"findex_24_{x}")
df_2024['survey_year'] = 2024
df_2024['data_source'] = 'Findex'
df_2024['hh_income'] = raw_findex_2024['inc_q']
df_2024['weight'] = (raw_findex_2024['wgt']).fillna(NA)

In [91]:
# econ rank construct
df_2010['hh_econ_rank'] = (df_2010.sort_values('hh_income')['weightslga'].cumsum() - 0.5 * df_2010['weightslga']) / df_2010['weightslga'].sum() * 100
df_2015['hh_econ_rank'] = (df_2015.sort_values('hh_income')['hh_weight'].cumsum() - 0.5 * df_2015['hh_weight']) / df_2015['hh_weight'].sum() * 100
df_2020['hh_econ_rank'] = (df_2020.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2020['weight']) / df_2020['weight'].sum() * 100
df_2021['hh_econ_rank'] = (df_2021.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2021['weight']) / df_2021['weight'].sum() * 100
df_2024['hh_econ_rank'] = (df_2024.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2024['weight']) / df_2024['weight'].sum() * 100